# Prácica 5.1.1: Redes Neuronales Recurrentes (Simples)

## Configuración del entorno

### Carga de librerías necesarias.

In [ ]:
from collections import Counter
import string
import numpy as np

# Descargar recursos de NLTK (si no se han descargado previamente)
import nltk
from nltk.tokenize import word_tokenize
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=False)

import torch
import torch.nn as nn
# Vemos si tenemos GPU disponible ...
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("Usando dispositivo: CUDA 🚀")
else:
    print("Usando dispositivo: CPU 😞")

# Dataset que vamos a utilizar.
DATASET= "dataset_conversational.txt"

### Definición de algunas funciones auxiliares

In [ ]:
# Función para cargar el dataset desde el fichero
def cargar_datos(archivo):
    with open(archivo, 'r', encoding='utf-8') as f:
        data = f.read().splitlines()
    return data

# Función para preprocesar una frase: Simplemente tokeniza.
def preprocesar_texto(texto):
    tokens = word_tokenize(texto.lower())  # Tokeniza y convierte a minúsculas
    # Eliminar signos de puntuación
    # tokens = [token for token in tokens if token not in string.punctuation]
    return tokens

### Definición de los hiper parámetros de entrenamiento del modelo

In [ ]:
# Hiperparámetros del modelo
input_size = 0                  # Tamaño del vocabulario (número de palabras únicas)
output_size = 0                 # El output es el tamaño del vocabulario (para clasificar cada palabra)
embedding_dim = 100             # Dimensión del embedding
activation_function = 'tanh'    # Función de activación (tanh o relu)
hidden_size = 64                # Número de dimensiones ocultas en la RNN
num_layers = 2                  # Número de capas recurrentes
learning_rate = 0.001           # Tasa de aprendizaje para el optimizador
batch_size = 10                 # Tamaño del batch durante el entrenamiento
seq_length = 2                  # Longitud máxima de las secuencias (contexto)
num_epochs = 200                # Número total de épocas de entrenamiento

## Pre Procesamiento del dataset
Realiza el preproceso de los datos de entrenamiento y los "formaliza" para que PyTorch pueda leerlos y entrenar la RNN.

In [ ]:
# Cargar el dataset (ajustar la ruta del archivo)
data = cargar_datos(DATASET)

# Crear un vocabulario (contando todas las palabras del corpus)
voc_size = len(Counter([palabra for linea in data for palabra in preprocesar_texto(linea)]))
print(f"Vocabulario de tamaño: {voc_size}")

# Asignar índices a cada palabra
word2idx = {}
for idx, palabra in enumerate(sorted(list(set([palabra for linea in data for palabra in preprocesar_texto(linea)])))):
    word2idx[palabra] = idx

# Convertir el texto en secuencias de números (índices)
def convertir_a_indices(tokens_linea):
    return [word2idx.get(token, 0) for token in tokens_linea]

# Crear listas de secuencias y etiquetas
secuencias = []
etiquetas = []

# Generar secuencias de entrenamiento
for linea in data:
    if len(linea.strip()) == 0:  # Saltar líneas vacías
        continue        
    tokens_linea = preprocesar_texto(linea)
    # Si la frase es demasiado corta, no podemos formar una secuencia de entrenamiento
    if len(tokens_linea) < seq_length + 1:
        continue
    # Convertir tokens a índices
    indices_linea = convertir_a_indices(tokens_linea)
    
    for i in range(len(indices_linea) - seq_length):
        secuencia = indices_linea[i:i+seq_length]
        etiqueta = indices_linea[i+seq_length]  # La palabra siguiente (como índice)
        secuencias.append(secuencia)
        etiquetas.append(etiqueta)

# Crear el dataset y dataloader para PyTorch
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels
        
    def __len__(self):
        return len(self.sequences)
        
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx]), torch.tensor([self.labels[idx]])
    
# Crear dataset y dataloader
dataset = TextDataset(secuencias, etiquetas)
dataloader = torch.utils.data.DataLoader(
    dataset,                    # Crear el dataset y dataloader para PyTorch
    batch_size= batch_size,      # Tamaño del batch durante el entrenamiento.
    shuffle= True                # Mezclar los datos en cada época.
)

# Muestra datos de ejemplo
# for i, (seq, label) in enumerate(dataloader):
#     print(f"Batch {i+1}")
#     print("Secuencias (input):", seq)
#     print("Etiquetas (target):", label)
#     if i == 1:  # Mostrar solo los primeros 2 batches
#         break

## Entrenamiento de la RNN

- `embedding`: Convierte índices de palabras en vectores densos.
- `rnn`: Define la arquitectira interna de nuestra Red Neuronal Recurrente.
- `fc`: Implementa la función linear $h_t = f(W_{hh}h_{t-1} + W_{xh}x_t + b_h)$
- `criterion`: La función de pérdida $L$. En este caso se utilizará 'Entropía Cruzada'.
- `optimizer`: El encargado de la actualización de los diferentes pasos en el proceso BPTT. En nuestro caso utilizaremos un optimizador Adam.

In [ ]:
# Asegurarse de que output_size esté acorde al vocabulario
output_size = voc_size

embedding = nn.Embedding(voc_size, embedding_dim)  # Capa de embeeding (Whx)

rnn = nn.RNN(
    input_size=embedding_dim,           # Dimensión de entrada (embedding_dim)
    hidden_size=hidden_size,            # Dimensión oculta
    num_layers=num_layers,              # Número de capas recurrentes
    batch_first=True,                   # Formato de los datos: (batch, seq)
    nonlinearity= activation_function   # Función de activación (tanh o relu)
)

fc = nn.Linear(hidden_size, output_size) # h_t = f(W_{hh}h_{t-1} + W_{xh}x_t + b_h)

# Mover módulos al dispositivo
embedding.to(device)
rnn.to(device)
fc.to(device)

# Función para definir la pérdida y el optimizador
criterion = nn.CrossEntropyLoss()   # Función de Pérdida L (Entropía Cruzada).

def init_weights_module(m):
    """Inicializar pesos con valores pequeños aleatorios"""
    if isinstance(m, nn.Linear):
        torch.nn.init.normal_(m.weight.data, mean=0, std=1)
        torch.nn.init.normal_(m.bias.data, 0.05)

# Inicalizar pesos
fc.apply(init_weights_module)

# Crear optimizador (Adam) sobre los parámetros de las tres componentes
optimizer = torch.optim.Adam(
    list(embedding.parameters()) + list(rnn.parameters()) + list(fc.parameters()), 
    lr=learning_rate
)

# Entrenamiento por batches y épocas (usando los módulos separados)
for epoch in range(num_epochs):
    total_loss = 0
    
    for batch_idx, (seqs, labels) in enumerate(dataloader):
        seqs = seqs.to(device)      # Mover datos al dispositivo
        labels = labels.to(device)  # Mover etiquetas al dispositivo
        
        # FORWARD PASS
        embedded = embedding(seqs)                  # Recuperar embeddings.
        output, hidden = rnn(embedded, None)        # Calcular salida y estado oculto en el instante t.
        logits = fc(output[:, -1])                  # Introducimos en la función lineal el output del último paso temporal.
        
        # Calcular pérdida entre predicciones y etiquetas verdaderas
        loss = criterion(logits, labels.squeeze())
        total_loss += loss.item()
        
        # Backward pass: Propagar el error y actualizar pesos
        optimizer.zero_grad() # Limpiar gradientes previos
        loss.backward() # Propagar hacia atrás
        for param in list(embedding.parameters()) + list(rnn.parameters()) + list(fc.parameters()):
            if param.grad is not None:
                # Evitar gradientes explosivos o desvanecientes.
                # Limita los gradientes a un rango razonable.
                param.grad.data.clamp_(-1e9, 1e9) 
        
        optimizer.step() # Actualizar pesos
    
    # Muestra la ganancia en cada epoch
    print(f'Epoch {epoch+1} de {num_epochs}, Loss: {total_loss/len(dataloader):.4f}')

## Guardado del Modelo

Guardamos el estado completo de la Red Neuronal para poder recuperarla posteriormente y usarla.

- `embedding.state_dict()`: Guarda los pesos del embedding layer. Este layer convierte índices de palabras (números) en vectores densos de 100 dimensiones. Es una matriz de tamaño (vocabulario, embedding_dim) que mapea cada palabra a su representación numérica.

- `rnn.state_dict()`: Guarda los pesos internos de la RNN. Incluye:
    - Matrices de peso: $W_hh$ (conexiones entre estados ocultos), $W_xh$ (entrada a estados ocultos
    - Sesgos $b$ (términos de sesgo para cada capa recurrente)
    - Se guarda el estado de todas las capas que componen la RNN.
- `fc.state_dict()`: Guarda los pesos de la capa fully connected (lineal). Esta capa transforma el último estado oculto de la RNN (128 dimensiones) en predicciones del vocabulario (voc_size dimensiones).
- `optimizer.state_dict()`:Guarda el estado del optimizador Adam. Contiene:
    - Información de momentos (media y varianza) acumulados durante el entrenamiento
    - Permite reanudar el entrenamiento desde exactamente donde se paró. Sin esto, si reentrenamos, Adam tendría que recalcular desde cero
- `loss`: Guarda el valor de pérdida final del último epoch. Útil para registrar cómo de bien se entrenó el modelo.
- `modelo_rnn_direct.pth`: Es el nombre del archivo donde se almacenan todos estos datos. El formato .pth es estándar en PyTorch (serialización con pickle).

Adicionalmente se ha guardado tres elementos más relativos al vocabulario utilizado para poder realizar inferencias y reentrenamientos posteriores:

- `word2idx`: Mapeo de palabras → índices

- `idx2word`: Mapeo de índices → palabras

- `voc_size`: Tamaño del vocabulario

In [ ]:
MODEL_NAME= 'modelo_rnn_direct.pth'

# Crear idx2word (mapeo inverso del vocabulario)
idx2word = {idx: palabra for palabra, idx in word2idx.items()}

torch.save(
    {
        'embedding_state': embedding.state_dict(), # Estado del embedding
        'rnn_state': rnn.state_dict(),              # Estado de la RNN
        'fc_state': fc.state_dict(),            # Estado de la capa fully connected
        'optimizer_state': optimizer.state_dict(),  # Estado del optimizador
        'loss': loss,                           # Pérdida final del entrenamiento
        'word2idx': word2idx,                   # Mapeo de palabras a índices
        'idx2word': idx2word,                   # Mapeo de índices a palabras
        'voc_size': voc_size,                   # Tamaño del vocabulario
    },
    MODEL_NAME
)
print("Modelo guardado como 'modelo_rnn_direct.pth'")

## Cargando el Modelo
A continuación se define una función para la carga de un modelo guradado de la manera que vimos en el paso anterior.

El objetivo de esta función es leer el archivo que guardamos en el pas anterior y restaurar los pesos de los componentes del modelo (embedding, RNN, capa lineal) así como el estado del optimizador para poder continuar con inferencia o reanudar entrenamiento.

- `checkpoint = torch.load('modelo_rnn_direct.pth', map_location=device)`
    - `torch.load(...)` deserializa el objeto guardado con torch.save.
    - `map_location=device` indica a PyTorch dónde colocar los tensores al cargarlos (por ejemplo cpu o cuda). Esto evita errores si el checkpoint fue guardado en GPU y ahora solo hay CPU, o viceversa.

- `embedding.load_state_dict(checkpoint['embedding_state'])`
    - `load_state_dict()` carga los parámetros (weights y bias) en el módulo embedding. Hace la asignación de tensores a los parámetros correspondientes por nombre.
    - Si faltan claves o hay incompatibilidades de forma, PyTorch lanzará un error o advertencia (se puede usar `strict=False` para permitir discrepancias).

- `rnn.load_state_dict(checkpoint['rnn_state'])`: Contendrá las matrices y sesgos recurrentes para todas las capas y estados.

- `fc.load_state_dict(checkpoint['fc_state'])`: Carga los pesos de la capa lineal final.

- `optimizer.load_state_dict(checkpoint['optimizer_state'])`: Restaura el estado del optimizador (importante para reanudar entrenamiento manteniendo la dinámica del optimizador).

También se tiene en cuenta la recuperación desde el checkpoint guardado de las colecciones y datos relativos al vocabulario utilizado durante el entrenamiento.

La función finalmente devuelve una tupla con los diferentes módulos cargados para poder reasignarlas a variables desde nuestro código.

In [ ]:
# Cargar el modelo para su uso posterior
def cargar_modelo(nombre_archivo= None):
    """
    Carga el modelo RNN guardado junto con el vocabulario necesario para inferencia y reentrenamiento.
    
    Returns:
        tuple: (embedding, rnn, fc, optimizer, word2idx, idx2word, voc_size)
            - embedding: Capa de embedding
            - rnn: Capa RNN
            - fc: Capa fully connected
            - optimizer: Optimizador Adam
            - word2idx: Diccionario que mapea palabras a índices
            - idx2word: Diccionario que mapea índices a palabras
            - voc_size: Tamaño del vocabulario
    """
    checkpoint = torch.load('modelo_rnn_direct.pth', map_location=device)
    embedding.load_state_dict(checkpoint['embedding_state'])
    rnn.load_state_dict(checkpoint['rnn_state'])
    fc.load_state_dict(checkpoint['fc_state'])
    optimizer.load_state_dict(checkpoint['optimizer_state'])
    
    # Cargar el vocabulario
    word2idx_loaded = checkpoint['word2idx']
    idx2word_loaded = checkpoint['idx2word']
    voc_size_loaded = checkpoint['voc_size']
    
    return embedding, rnn, fc, optimizer, word2idx_loaded, idx2word_loaded, voc_size_loaded

## Pruebas sobre la RNN
Vamos a realizar una prueba de predicción:

1. Cargamos el modelo de la RNN guardado en disco previamente.
2. Introducimos una secuencia inicial para que la RNN la complete. Podemos precisar cuántas _completions_ va a hacer por medio de la variable `num_palabras_a_generar`

In [ ]:
# Cargamos el modelo guardado.
embedding, rnn, fc, optimizer, word2idx, idx2word, voc_size = cargar_modelo(nombre_archivo=MODEL_NAME)

# Recoge una palabra y ejecuta la predicción 200 veces incorporando la predicción a la secuencia y volviendo
# a predecir la siguiente palabra en bucle hasta que se completen las num_palabras iteraciones
# secuencia_inicial = "¿Qué harías"
secuencia_inicial = "¿Qué consejo"
num_palabras_a_generar = 10
texto_generado = secuencia_inicial

for _ in range(num_palabras_a_generar):
    # Preparar la secuencia de entrada
    tokens_iniciales = word_tokenize(texto_generado.lower())
    indices_iniciales = [word2idx.get(token, 0) for token in tokens_iniciales]
    
    # Asegurarse de que la secuencia tiene la longitud adecuada
    if len(indices_iniciales) < seq_length:
        indices_iniciales = [0] * (seq_length - len(indices_iniciales)) + indices_iniciales
    else:
        indices_iniciales = indices_iniciales[-seq_length:]
    
    # Convertir a tensor y mover al dispositivo
    input_tensor = torch.tensor([indices_iniciales]).to(device)

    # Modo evaluación
    embedding.eval(); rnn.eval(); fc.eval()

    # Realizar la predicción
    with torch.no_grad():
        embedded = embedding(input_tensor)
        output, hidden = rnn(embedded, None)
        logits = fc(output[:, -1])
        _, predicted_index = torch.max(logits, dim=1)
        siguiente_palabra_idx = predicted_index.item()
        palabra_siguiente = idx2word.get(siguiente_palabra_idx, '<UNK>')
        texto_generado += " " + palabra_siguiente
print(f"Texto generado: {texto_generado}")